# HR / Policy Chatbot (Upload Any Document)

This notebook builds a conversational chatbot that can answer questions about **any uploaded HR/policy document** using a Retrieval-Augmented Generation (RAG) pipeline:

- Extract text from **PDF / DOCX / TXT**
- Split into overlapping chunks
- Create embeddings for each chunk
- Retrieve the most relevant chunks for each question
- Generate a grounded answer with a chat model

> Note: This is a simple, notebook-friendly RAG implementation (in-memory embeddings).

## Step 1: Install Required Libraries

We install the key packages:
- `openai` — to access GPT and embedding models
- `gradio` — to build the chat interface
- `numpy` — for vector math (cosine similarity)

In [ ]:
# Install required packages
%pip install openai gradio numpy pypdf python-docx --quiet

## Step 2: Import Libraries

In [ ]:
import os
import pathlib
from typing import List, Tuple, Optional

import openai
import numpy as np
import gradio as gr
from pypdf import PdfReader
from docx import Document

print("Libraries imported successfully!")

## Step 3: Set Your OpenAI API Key

Replace `"your-api-key-here"` with your actual OpenAI API key.

> ⚠️ **Security tip:** Never share your API key publicly or commit it to version control.

In [ ]:
# Set your OpenAI API key and initialize the client
# Recommended: set environment variable OPENAI_API_KEY instead of hardcoding.
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or "your-api-key-here"  # <-- Replace if not using env var

client = openai.OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI client initialized successfully!")

## Step 4: Load **Any** HR/Policy Document

Instead of hardcoding one document, we'll load **any** uploaded file (PDF/DOCX/TXT), extract all its text, and build the chatbot's knowledge base from it.

Supported formats:
- `.pdf`
- `.docx`
- `.txt`

In [ ]:
def _normalize_text(text: str) -> str:
    # Keep it simple but consistent.
    return "\n".join(line.rstrip() for line in text.replace("\r\n", "\n").replace("\r", "\n").split("\n")).strip()


def load_document_text(file_path: str) -> str:
    """Load a document from disk and return extracted text.

    Supported: PDF, DOCX, TXT
    """
    path = pathlib.Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    suffix = path.suffix.lower()

    if suffix == ".pdf":
        reader = PdfReader(str(path))
        pages = []
        for page in reader.pages:
            pages.append(page.extract_text() or "")
        return _normalize_text("\n\n".join(pages))

    if suffix == ".docx":
        doc = Document(str(path))
        parts: List[str] = []
        for p in doc.paragraphs:
            if p.text and p.text.strip():
                parts.append(p.text)
        return _normalize_text("\n".join(parts))

    if suffix == ".txt":
        return _normalize_text(path.read_text(encoding="utf-8", errors="replace"))

    raise ValueError(f"Unsupported file type: {suffix}. Please upload a PDF, DOCX, or TXT.")


print("Document loader ready. You'll select a file in the UI step below.")

## Step 5: Split the Document into Chunks

We break the document into smaller overlapping chunks. This is important because:
- Embedding models work better on shorter, focused pieces of text
- Overlap ensures we don't lose context at chunk boundaries

**Parameters:**
- `chunk_size=600` — each chunk is ~600 characters
- `overlap=100` — consecutive chunks share 100 characters of context

In [ ]:
def chunk_text(text: str, chunk_size: int = 1200, overlap: int = 200) -> List[str]:
    """Splits a long text into overlapping chunks."""
    chunks: List[str] = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += max(1, chunk_size - overlap)

    return chunks


# These globals will be filled after you upload a document.
document_name: Optional[str] = None
source_text: Optional[str] = None
chunks: List[str] = []
chunk_embeddings: List[List[float]] = []

print("Chunker ready. Upload a document to create chunks/embeddings.")

## Step 6: Generate Embeddings for Each Chunk

We convert every chunk into a numerical vector (embedding) using OpenAI's `text-embedding-ada-002` model. These vectors capture the **semantic meaning** of each chunk — similar ideas will have similar vectors.

> ⏳ This may take a moment depending on the number of chunks.

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"


def get_embeddings(texts: List[str], model: str = EMBEDDING_MODEL, batch_size: int = 64) -> List[List[float]]:
    """Embed many texts efficiently (batched API calls)."""
    out: List[List[float]] = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        resp = client.embeddings.create(input=batch, model=model)
        out.extend([d.embedding for d in resp.data])
    return out


def build_knowledge_base(file_path: str) -> Tuple[str, str, List[str], List[List[float]]]:
    """Load a file, chunk it, and build embeddings."""
    text = load_document_text(file_path)
    local_chunks = chunk_text(text, chunk_size=1200, overlap=200)
    if not local_chunks:
        raise ValueError("No text could be extracted from this file.")

    print(f"Chunking complete: {len(local_chunks)} chunks")
    print("Generating embeddings... (this may take a moment)")
    local_embeddings = get_embeddings(local_chunks)
    print(f"Embeddings complete: {len(local_embeddings)} vectors")

    return pathlib.Path(file_path).name, text, local_chunks, local_embeddings


print("Embedding/index builder ready. You'll run it automatically from the UI.")

## Step 7: Build the Similarity Search Function

When a user asks a question, we:
1. Convert the question to an embedding vector
2. Compare it against all chunk embeddings using **cosine similarity**
3. Return the top-K most relevant chunks as context

**Cosine similarity** measures the angle between two vectors — a score of 1.0 means identical meaning, 0.0 means unrelated.

In [ ]:
def cosine_similarity(vec_a: List[float], vec_b: List[float]) -> float:
    a = np.asarray(vec_a, dtype=np.float32)
    b = np.asarray(vec_b, dtype=np.float32)
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def find_relevant_chunks(query: str, top_k: int = 4) -> List[str]:
    if not chunks or not chunk_embeddings:
        return []

    # Embed the query
    query_embedding = get_embeddings([query])[0]

    # Score all chunks
    similarities = [cosine_similarity(query_embedding, emb) for emb in chunk_embeddings]
    top_indices = np.argsort(similarities)[-top_k:][::-1]

    return [chunks[i] for i in top_indices]


print("Retriever ready.")

## Step 8: Build the Answer Generation Function

We combine the retrieved context with the user's question and send it to GPT-3.5-Turbo using a structured STAR-style prompt:
- **S**ituation: You are a Nestlé HR assistant
- **T**ask: Answer questions using only the provided HR policy
- **A**ction: Retrieve relevant context, construct prompt, call GPT
- **R**esult: A grounded, professional answer

The function also supports **multi-turn conversation** by passing chat history to GPT.

In [ ]:
def generate_answer(question: str, chat_history: List[Tuple[str, str]]) -> str:
    if not chunks or not chunk_embeddings:
        return "Please upload a policy/HR document first so I can answer based on it."

    relevant_chunks = find_relevant_chunks(question, top_k=4)
    context = "\n\n".join(relevant_chunks)

    system_prompt = f"""You are a professional and friendly HR Assistant.

You must answer the user's question using ONLY the provided document excerpts.
- If the answer is present, explain it clearly and professionally.
- If the answer is not present, say you don't have enough information from the document and suggest what to do next.
- Keep responses concise (2–6 sentences) unless the user asks for more detail.

Document name: {document_name or "(unknown)"}

--- Document Excerpts ---
{context}
-------------------------"""

    messages = [{"role": "system", "content": system_prompt}]

    for user_msg, assistant_msg in chat_history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})

    messages.append({"role": "user", "content": question})

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=0.2,
        max_tokens=600,
    )

    return response.choices[0].message.content


print("Answer generation function defined!")

## Step 9: Quick Test Before Launching the UI

Let's verify the full pipeline works end-to-end with a sample question before opening the Gradio interface.

In [ ]:
# Optional: after uploading a document in the UI, you can test here
# test_question = "How does the company evaluate employee performance?"
# print(generate_answer(test_question, []))

print("Run the UI cell below, upload a document, then chat.")

## Step 10: Launch the Gradio Chatbot Interface

We now build a polished, interactive chat UI using Gradio's `ChatInterface`. Users can:
- Type any HR-related question
- Click example questions to get started quickly
- Have a multi-turn conversation with memory of previous messages

> 🌐 Running this cell will open the chatbot in your browser (or show a public link if on a remote server).

In [ ]:
import gradio as gr
from typing import List, Tuple


def chatbot_response(message: str, history: List[Tuple[str, str]]) -> str:
    return generate_answer(message, history)


def on_file_uploaded(file_obj):
    """Build KB from uploaded file and store it in globals."""
    global document_name, source_text, chunks, chunk_embeddings

    if file_obj is None:
        document_name = None
        source_text = None
        chunks = []
        chunk_embeddings = []
        return "No file uploaded yet."

    # Gradio gives a tempfile path-like object
    file_path = getattr(file_obj, "name", None) or str(file_obj)

    document_name, source_text, chunks, chunk_embeddings = build_knowledge_base(file_path)
    return f"Loaded '{document_name}'. Chunks: {len(chunks)}. Ready to chat."


with gr.Blocks() as demo:
    gr.Markdown("## HR / Policy Chatbot (Upload any document)")
    gr.Markdown(
        "Upload a **PDF, DOCX, or TXT**. The app will read the entire document, build embeddings, and then answer questions grounded in that document."
    )

    file_in = gr.File(label="Upload document", file_types=[".pdf", ".docx", ".txt"], type="filepath")
    status = gr.Textbox(label="Status", interactive=False)

    file_in.change(fn=on_file_uploaded, inputs=[file_in], outputs=[status])

    chat = gr.ChatInterface(
        fn=chatbot_response,
        title="HR Assistant",
        description="Ask questions about the uploaded document.",
        examples=[
            "Summarize the key HR principles in this document.",
            "What does the document say about performance management?",
            "What are the rules on harassment/discrimination?",
            "What does it say about training and development?",
        ],
        cache_examples=False,
    )

# Launch the app
# Set share=True to generate a public link you can share with others
# prevent_thread_lock=True lets this cell finish (useful for automated runs)
demo.launch(share=False, prevent_thread_lock=True)

---
## Summary

| Step | What we did | Why it matters |
|------|-------------|----------------|
| 1 | Installed libraries | Sets up the Python environment |
| 2-3 | Imported libraries & API key | Connects to OpenAI services |
| 4 | Loaded HR Policy document | Provides the knowledge base |
| 5 | Split text into chunks | Makes embeddings accurate and focused |
| 6 | Generated embeddings | Converts text to searchable vectors |
| 7 | Built similarity search | Retrieves only relevant policy sections |
| 8 | Built answer generation | Uses GPT to give natural, grounded answers |
| 9 | Tested the pipeline | Verified everything works before the UI |
| 10 | Launched Gradio UI | Created the interactive chatbot interface |

### Architecture: RAG (Retrieval-Augmented Generation)
This chatbot uses the **RAG** pattern — instead of relying solely on GPT's training knowledge, it first *retrieves* relevant facts from the HR document, then *augments* the GPT prompt with that context. This makes the answers more accurate and grounded in Nestlé's actual policies.